In [2]:
#IMPORT LIBRARIES     
import os
import pandas as pd
import numpy as np
import string

In [3]:
#LOAD DATSET
df=pd.read_csv("../../res/ddos_dataset.csv", low_memory=False)  


In [4]:
#REMOVE IMPOSSIBLE ROW
df = df.drop(index=20950).reset_index(drop=True)


In [5]:
#REMOVE SAME NAME CORRUPTED COLUMNS: INDEX 41 AND 62

print(df.shape)

df = df.drop(df.columns[41], axis=1)
df = df.drop(df.columns[61], axis=1)  #NOTE: THE INDEX CHANGE FROM 62 TO 61

print(df.shape)


(64238, 88)
(64238, 86)


In [6]:
#CLEAN COLUMN NAMES

#Function to clear a name column
def clean_column_name(name):   
    
    name = name.strip() #Remove spaces to the beginnig e to the end of the word(not in the middle!)
    name=name.lower()   #Convert all to lower_case
    
    allowed = set(string.ascii_lowercase + string.digits + "_/")  #Build set of allow caracters 

    cleaned_name = ""   #name_cleaned
    for char in name:
        if char in allowed:
            cleaned_name += char      #Leave that char
        else:
            cleaned_name += "_"       #Convert to underscore

    
    while "__" in cleaned_name:
        cleaned_name = cleaned_name.replace("__", "_") #Remove consecutive underscores

    cleaned_name = cleaned_name.strip("_") #Remove underscore to the begining and at the end

    return cleaned_name

# Clean column names
df.columns = df.columns.astype(str) #Convert all column names on string
new_columns = []
for original_col_name in df.columns:
    new_name = clean_column_name(original_col_name)
    new_columns.append(new_name)

df.columns = new_columns


In [7]:
#REMOVE UNNAMED COLUMNS 

df = df.loc[:, df.columns != ""]   #Keep only columns that have a name
df=df.loc[:, ~df.columns.str.startswith("unnamed")]  #Remove columns unnamed0, unnamed1,...


In [8]:
#CONVERSION TYPE TO INT64 OR FLOAT64 OF NUMERICAL COLUMNS TO INT64 OR FLOAT64

columns_str = [           # String_column to not tuch 
    "flow_id",
    "source_ip",
    "destination_ip",
    "timestamp",
    "label",
]


columns_numerical=[]      #Select numerical columns in df

for column in df.columns:      
    if column not in columns_str:
        columns_numerical.append(column) 

for column in columns_numerical:   #Create a copy_work of numerical columns
    prov_column = df[column]    

    
    prov_column_str = prov_column.astype(str)        #Convert to string to replace ',' with '.' (after)
    prov_column_str = prov_column_str.str.strip()    #Remove spaces to left and right
    prov_column_str = prov_column_str.str.replace(",", ".", regex=False)     #Replace ',' with '.'

    df[column] = pd.to_numeric(prov_column_str, errors="coerce")  #Convert all numerical columns values into a number int64 or float 64
    #    - "inf", "-inf" become inf and -inf
    #    - not convertible value become NaN
    #We wait to convert on float because we use these type to classify our columns (Serves after)


In [9]:
#PUT HERE ALL PLOTS VISUALISATION

In [10]:
#REMOVE COLUMNS WITH NAN OR IMPOSSIBLE NEGATIVE VALUES
to_remove=['simillarhttp', 'init_win_bytes_backward',
           'init_win_bytes_forward','min_seg_size_forward'] 
df = df.drop(columns=to_remove)
for col in to_remove:
    columns_numerical.remove(col)


In [11]:
#ADD NEW FEATURES

df['total_packets'] = df['total_fwd_packets'] + df['total_backward_packets']

df['total_bytes'] = df['total_length_of_fwd_packets'] + df['total_length_of_bwd_packets']



df['direction_imbalance'] = (df['total_fwd_packets'] - df['total_backward_packets']) / (
                             df['total_fwd_packets'] + df['total_backward_packets'] + 1)


flag_cols = ['fwd_psh_flags','fwd_urg_flags','fin_flag_count','syn_flag_count','rst_flag_count']
present_flags = [c for c in flag_cols if c in df.columns] 
df['flag_sum'] = df[present_flags].sum(axis=1) 

df['flag_fraction'] = df['flag_sum'] / (df['total_packets'] + 1)
df.drop(columns=['flag_sum'], inplace=True) #we need only to build flag fraction



df['fwd_bytes_fraction'] = df['total_length_of_fwd_packets'] / (df['total_bytes'] + 1)


df['is_unidirectional'] = (
    (df['total_fwd_packets'] == 0) | (df['total_backward_packets'] == 0)
).astype(int)



df['fwd_data_fraction'] = df['act_data_pkt_fwd'] / (df['total_fwd_packets'] + 1)

df['ack_fraction'] = df['ack_flag_count'] / (df['total_packets'] + 1)


den = df['flow_iat_mean'].replace(0, np.nan)
df['flow_iat_cv'] = (df['flow_iat_std'] / den).fillna(0)

den = df['packet_length_mean'].replace(0, np.nan)
df['packet_len_cv'] = (df['packet_length_std'] / den).fillna(0)

list_new_add_feature=['fwd_data_fraction',
'ack_fraction',
'flow_iat_cv',
'packet_len_cv',
'direction_imbalance',
'flag_fraction',
'fwd_bytes_fraction',
'is_unidirectional',
'total_packets',
'total_bytes']

#Adding to columns_numerical
columns_numerical=columns_numerical + list_new_add_feature


In [12]:
#PUT HERE ALL PLOTS OF NEW FEATURES

In [13]:
#CLASSIFICATION OF COLUMNS BASED ON BELONG GROUP

columns_str           #Already classified 
columns_numerical     #Already classified

columns_numerical_categorical=['source_port',  #They are part of columns_numerical
                               'destination_port', 
                               'protocol',
                               'inbound']


In [14]:
#REMOVE CONSTANT COLUMNS

print(df.shape)

def drop_constant_columns(df, cols_to_check):
    
    constant_cols =[]
    for col in cols_to_check:
        if df[col].nunique() == 1:
            constant_cols.append(col)
            
    df_clean = df.drop(columns=constant_cols)
    
    non_constant_cols = [col for col in cols_to_check if col not in constant_cols]
   
    return df_clean, non_constant_cols

#REMOVE
df, columns_numerical = drop_constant_columns(df,columns_numerical)

print(df.shape) 


(64238, 91)
(64238, 79)


In [15]:
#FIRST PROBLEM: CODIFY CATEGORICAL FEATURE
#HERE THERE IS DATA LEAK BUT IF THEY WILL BE DIFFERET THERE WILL 
#PROBLEM TO COMPAIR THE TWO SPLITS



In [16]:
#FEATURE ENCODING FOR CATEGORICAL COLUMNS

#INBOUND: ALREADY GOOD

# PROTOCOL: ONE-HOT ENCODING
protocol_ohe = pd.get_dummies(df["protocol"], prefix="protocol", dtype=int)

df = df.drop(columns=["protocol"])
df = df.join(protocol_ohe)
df = df.drop(columns=["protocol_0"])  #Avoid dummy variable trap


#SOURCE_PORT: BUCKETING; IT IS NOT DATALEAK BECAUSE SOURCE_PORT CAN BE RANDOM

df["src_port_low"] = (df["source_port"] < 1024).astype(int)      # 0–1023
df["src_port_medium"]     = ((df["source_port"] >= 1024) & (df["source_port"] < 49152)).astype(int)  # 1024–49151
df["src_port_high"] = (df["source_port"] >= 49152).astype(int)                                       # 49152–65535

df = df.drop(columns=["source_port"])

df = df.drop(columns=["src_port_low"])    #Avoid dummy variable trap



#DESTINATION_PORT: TOP-K + ONE HOT
COL = "destination_port"
K = 6

topk = df[COL].value_counts().head(K).index

dport_topk = df[COL].where(df[COL].isin(topk), "other")

dport_num = pd.get_dummies(dport_topk, prefix="dport", dtype=int)

df=df.drop(columns=[COL]).join(dport_num)  #here drop destination_port
df=df.drop(columns=["dport_other"])        #avoid dummy variable trap



columns_numerical_categorical=["dport_0",
                               "dport_22",
                               "dport_53", 
                               "dport_80",
                               "dport_443",
                               "dport_465",                              
                               "src_port_medium",
                               "src_port_high", 
                               "protocol_6",
                               "protocol_17",
                               "inbound"]

#I don't update columns_numerical because since now we don't need


In [17]:
#CONVERTING LABEL TO NUMBERS
df['label'] = pd.Categorical(df['label']).codes


In [18]:
#CONVERT TO CSV
df.to_csv("data_set_after_clean.csv", index=False)

print(df.shape)

(64238, 86)


In [ ]:
#normalization, corr (modified, not choosing based on label), pca (make sure not using label)
#df.to_csv("dataset_for_clustering")

In [ ]:
# creating dataset for clustering